In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "31cfad8b",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "application/javascript": [
       "\n",
       "try {\n",
       "require(['notebook/js/codecell'], function(codecell) {\n",
       "  codecell.CodeCell.options_default.highlight_modes[\n",
       "      'magic_text/x-csrc'] = {'reg':[/^%%microblaze/]};\n",
       "  Jupyter.notebook.events.one('kernel_ready.Kernel', function(){\n",
       "      Jupyter.notebook.get_cells().map(function(cell){\n",
       "          if (cell.cell_type == 'code'){ cell.auto_highlight(); } }) ;\n",
       "  });\n",
       "});\n",
       "} catch (e) {};\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "application/javascript": [
       "\n",
       "try {\n",
       "require(['notebook/js/codecell'], function(codecell) {\n",
       "  codecell.CodeCell.options_default.highlight_modes[\n",
       "      'magic_text/x-csrc'] = {'reg':[/^%%pybind11/]};\n",
       "  Jupyter.notebook.events.one('kernel_ready.Kernel', function(){\n",
       "      Jupyter.notebook.get_cells().map(function(cell){\n",
       "          if (cell.cell_type == 'code'){ cell.auto_highlight(); } }) ;\n",
       "  });\n",
       "});\n",
       "} catch (e) {};\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Programming Weights...\n",
      "Packing Data...\n",
      "Setup is complete, buffer ready at phys_addr: 0x15900000\n"
     ]
    }
   ],
   "source": [
    "import os\n",
    "import time\n",
    "import subprocess\n",
    "import numpy as np\n",
    "import xml.etree.ElementTree as ET\n",
    "from pynq import Overlay, MMIO, allocate\n",
    "\n",
    "# --- Configuration ---\n",
    "BITFILE   = \"WNNAcceleratorBlk.bit\" \n",
    "DATA_FILE = \"mnist_fpga_test_data.npz\"\n",
    "LUT_DIR   = \"luts\"\n",
    "CPP_EXE   = \"wnn_runner\"\n",
    "\n",
    "NUM_LUTS         = 250\n",
    "N_CLASSES        = 10\n",
    "ADDR_BITS        = 6\n",
    "M                = 1 << ADDR_BITS \n",
    "LUT_DATA_WIDTH   = 8       \n",
    "WORDS_PER_ENTRY  = 4       \n",
    "INPUT_BITS       = 25088 \n",
    "DMA_TRANSFER_LEN = INPUT_BITS // 32 \n",
    "\n",
    "# --- 1. Load Hardware ---\n",
    "if not os.path.exists(BITFILE): raise FileNotFoundError(f\"Missing {BITFILE}\")\n",
    "overlay = Overlay(BITFILE) \n",
    "\n",
    "try:\n",
    "    dma = overlay.axi_dma_0\n",
    "    wnn_ip = overlay.wnn_axi_0 \n",
    "except AttributeError:\n",
    "    for name, ip in overlay.ip_dict.items():\n",
    "        if 'wnn' in name.lower(): wnn_ip = MMIO(ip['phys_addr'], ip['addr_range'])\n",
    "        if 'dma' in name.lower(): dma = getattr(overlay, name)\n",
    "if not wnn_ip or not dma: raise Exception(\"IPs not found!\")\n",
    "\n",
    "# --- 2. Helper for BRAM ---\n",
    "def get_bram_driver(overlay, bitfile_path):\n",
    "    keys = [k for k in overlay.ip_dict.keys() if 'bram' in k.lower()]\n",
    "    if keys: return overlay.ip_dict[keys[0]], overlay.ip_dict[keys[0]].mmio.length\n",
    "    \n",
    "    hwh_path = bitfile_path.replace(\".bit\", \".hwh\")\n",
    "    if not os.path.exists(hwh_path): return None, 0\n",
    "    tree = ET.parse(hwh_path)\n",
    "    for module in tree.getroot().iter('MODULE'):\n",
    "        if 'axi_bram_ctrl' in module.get('VLNV', '').lower():\n",
    "            base = int(next(p.get('VALUE') for p in module.iter('PARAMETER') if p.get('NAME') == 'C_S_AXI_BASEADDR'), 16)\n",
    "            high = int(next(p.get('VALUE') for p in module.iter('PARAMETER') if p.get('NAME') == 'C_S_AXI_HIGHADDR'), 16)\n",
    "            return MMIO(base, high-base+1), high-base+1\n",
    "    raise Exception(\"BRAM not found\")\n",
    "\n",
    "bram_ip, bram_size = get_bram_driver(overlay, BITFILE)\n",
    "\n",
    "# --- 3. Program Weights ---\n",
    "print(f\"Programming Weights...\")\n",
    "MAX_VAL = (1 << LUT_DATA_WIDTH) - 1 \n",
    "for l in range(NUM_LUTS):\n",
    "    mem_path = os.path.join(LUT_DIR, f\"lut_{l:03d}.mem\")\n",
    "    with open(mem_path, 'r') as f: lines = f.readlines()\n",
    "    \n",
    "    for addr_idx, line in enumerate(lines):\n",
    "        if addr_idx >= M: break\n",
    "        counts = [int(x, 16) for x in line.strip().split()] \n",
    "        \n",
    "        packed_val = 0\n",
    "        for c in range(N_CLASSES):\n",
    "            val = min(counts[c], MAX_VAL)\n",
    "            packed_val |= (val << (c * LUT_DATA_WIDTH))\n",
    "            \n",
    "        entry_offset = (l * M + addr_idx) * (WORDS_PER_ENTRY * 4)\n",
    "        for w in range(WORDS_PER_ENTRY):\n",
    "            chunk = (packed_val >> (w * 32)) & 0xFFFFFFFF\n",
    "            bram_ip.write(entry_offset + (w*4), chunk)\n",
    "\n",
    "# --- 4. Data Prep & Allocation ---\n",
    "print(\"Packing Data...\")\n",
    "data = np.load(DATA_FILE)\n",
    "x_test_bits = data['x'] \n",
    "y_test = data['y']      \n",
    "total_images = len(y_test)\n",
    "\n",
    "# Check if big_buffer already exists\n",
    "try:\n",
    "    if 'big_buffer' in locals():\n",
    "        big_buffer.freebuffer()\n",
    "except:\n",
    "    pass\n",
    "\n",
    "# Allocate contiguous memory for C++ access\n",
    "big_buffer = allocate(shape=(total_images * DMA_TRANSFER_LEN,), dtype=np.uint32)\n",
    "\n",
    "t_pack = time.time()\n",
    "for i in range(total_images):\n",
    "    packed_bytes = np.packbits(x_test_bits[i].astype(np.uint8), bitorder='little')\n",
    "    packed_words = np.frombuffer(packed_bytes, dtype=np.uint32)\n",
    "    \n",
    "    start_idx = i * DMA_TRANSFER_LEN\n",
    "    big_buffer[start_idx : start_idx + DMA_TRANSFER_LEN] = packed_words\n",
    "\n",
    "big_buffer.flush() \n",
    "y_test.astype(np.uint8).tofile(\"y_test.bin\")\n",
    "print(f\"Setup is complete, buffer ready at phys_addr: {hex(big_buffer.device_address)}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "237acb72",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Launching C++ Accelerator on 10000 images...\n",
      "--- C++ Output ---\n",
      "[C++] Starting Loop...\n",
      "\n",
      "[C++] FINAL REPORT:\n",
      "      Accuracy: 95.97% (9597/10000)\n",
      "      Time:     0.3828 s\n",
      "      FPS:      26124.01\n",
      "\n"
     ]
    }
   ],
   "source": [
    "# Execute Inference\n",
    "\n",
    "print(f\"Launching C++ Accelerator on {total_images} images...\")\n",
    "t_start = time.time()\n",
    "\n",
    "cmd = [\n",
    "    f\"./{CPP_EXE}\", \n",
    "    str(wnn_ip.mmio.base_addr), \n",
    "    str(dma.mmio.base_addr), \n",
    "    str(big_buffer.device_address), \n",
    "    str(total_images),\n",
    "    \"y_test.bin\"\n",
    "]\n",
    "\n",
    "proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)\n",
    "out, err = proc.communicate()\n",
    "\n",
    "t_end = time.time()\n",
    "print(\"--- C++ Output ---\")\n",
    "print(out)\n",
    "if err: print(\"STDERR:\", err)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "id": "830971d5",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Buffer freed.\n"
     ]
    }
   ],
   "source": [
    "# CLEAN UP\n",
    "if 'big_buffer' in locals():\n",
    "    big_buffer.freebuffer()\n",
    "    del big_buffer\n",
    "    print(\"Buffer freed.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "704bd470",
   "metadata": {},
   "outputs": [],
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.4"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}


Programming Weights...
Packing Data...
Packing complete in 9.42s

Launching C++ Accelerator...
[C++] Starting Loop...

[C++] FINAL REPORT:
      Accuracy: 95.12% (9512/10000)
      Time:     0.3836 s
      FPS:      26072.11

